# CS224N 作业 1：探索词向量（25 分）
### <font color='blue'> 截止时间：2024年4月9日星期二下午4:30</font>

欢迎来到 CS224N！

在开始之前，请确保你**阅读了与本笔记本同一目录下的 README.md**，以获取重要的环境设置信息。你需要安装一些 Python 库才能成功完成本次作业。本笔记本中提供了大量代码，我们强烈建议你阅读并理解它们，这也是学习的一部分 :)

如果你对 Python、Numpy 或 Matplotlib 不太熟悉，我们建议你参加周五的复习课程。该课程将被录播，相关资料将在我们的[网站](http://web.stanford.edu/class/cs224n/index.html#schedule)上提供。CS231N 的 Python/Numpy [教程](https://cs231n.github.io/python-numpy-tutorial/)也是一个很好的资源。


**作业说明：** 请确保在操作过程中及时保存笔记本。提交说明位于笔记本底部。

In [1]:
# All Import Statements Defined Here
# Note: Do not add to this list.
# ----------------
import sys
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 8

from platform import python_version
assert int(python_version().split(".")[1]) >= 5, "Please upgrade your Python version following the instructions in \
    the README.md file found in the same directory as this notebook. Your Python version is " + python_version()

from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

from datasets import load_dataset
imdb_dataset = load_dataset("stanfordnlp/imdb")

import re
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA

START_TOKEN = '<START>'
END_TOKEN = '<END>'
NUM_SAMPLES = 150

np.random.seed(0)
random.seed(0)
# ----------------

Matplotlib is building the font cache; this may take a moment.


ConnectTimeout: [Errno 60] Operation timed out

## 词向量

词向量通常作为下游 NLP 任务（如问答、文本生成、翻译等）的基础组件，因此建立对其优势和局限的直觉理解非常重要。在此，你将探索两种类型的词向量：基于*共现矩阵*的词向量和通过 *GloVe* 得到的词向量。

**术语说明：** "词向量"和"词嵌入"这两个术语经常互换使用。"嵌入"一词指的是我们在一个较低维度的空间中编码单词的语义信息。正如[维基百科](https://en.wikipedia.org/wiki/Word_embedding)所述，"*从概念上讲，它涉及从一个每个词一维的空间到一个连续的低维向量空间的数学嵌入*"。

## 第 1 部分：基于计数的词向量（10 分）

大多数词向量模型都源于以下思想：

*观其伴，知其意（[Firth, J. R. 1957:11](https://en.wikipedia.org/wiki/John_Rupert_Firth)）*

许多词向量实现都基于这样的理念：相似的词（即近义词）会出现在相似的语境中。因此，相似的词往往会与一组共同的词（即上下文）一起出现或使用。通过考察这些上下文，我们可以尝试为单词开发嵌入表示。基于这种直觉，许多"传统"的词向量构建方法依赖于词频统计。这里我们详细阐述其中一种策略——*共现矩阵*（更多信息请参见[此处](https://web.stanford.edu/~jurafsky/slp3/6.pdf)或[此处](https://web.archive.org/web/20190530091127/https://medium.com/data-science-group-iitr/word-embedding-2d05d270b285)）。

### 共现

共现矩阵统计的是事物在某种环境中共同出现的频率。给定文档中出现的某个词 $w_i$，我们考虑 $w_i$ 周围的*上下文窗口*。假设我们的固定窗口大小为 $n$，那么就是该文档中 $w_i$ 前后的 $n$ 个词，即单词 $w_{i-n} \dots w_{i-1}$ 和 $w_{i+1} \dots w_{i+n}$。我们构建一个*共现矩阵* $M$，这是一个对称的词-词矩阵，其中 $M_{ij}$ 表示在所有文档中，$w_j$ 出现在 $w_i$ 窗口内的次数。

**示例：固定窗口 n=1 的共现**：

文档 1："all that glitters is not gold"

文档 2："all is well that ends well"


|     *    | `<START>` | all | that | glitters | is   | not  | gold  | well | ends | `<END>` |
|----------|-------|-----|------|----------|------|------|-------|------|------|-----|
| `<START>`    | 0     | 2   | 0    | 0        | 0    | 0    | 0     | 0    | 0    | 0   |
| all      | 2     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| that     | 0     | 1   | 0    | 1        | 0    | 0    | 0     | 1    | 1    | 0   |
| glitters | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| is       | 0     | 1   | 0    | 1        | 0    | 1    | 0     | 1    | 0    | 0   |
| not      | 0     | 0   | 0    | 0        | 1    | 0    | 1     | 0    | 0    | 0   |
| gold     | 0     | 0   | 0    | 0        | 0    | 1    | 0     | 0    | 0    | 1   |
| well     | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 1    | 1   |
| ends     | 0     | 0   | 1    | 0        | 0    | 0    | 0     | 1    | 0    | 0   |
| `<END>`      | 0     | 0   | 0    | 0        | 0    | 0    | 1     | 1    | 0    | 0   |

在自然语言处理中，我们通常使用 `<START>` 和 `<END>` 标记来表示句子、段落或文档的开头和结尾。这些标记被包含在共现计数中，围绕每个文档，例如："`<START>` All that glitters is not gold `<END>`"。

矩阵的行（或列）提供了基于词-词共现的词向量，但它们可能很大。为了降低维度，我们采用奇异值分解（SVD），类似于 PCA，选择前 $k$ 个主成分。SVD 过程将共现矩阵 $A$ 分解为对角矩阵 $S$ 中的奇异值和 $U_k$ 中新的更短的词向量。

这种降维方法保持了语义关系；例如，*doctor* 和 *hospital* 会比 *doctor* 和 *dog* 更接近。

对于那些不熟悉特征值和 SVD 的人，这里有一个适合初学者的 SVD 介绍，请参见[此处](https://davetang.org/file/Singular_Value_Decomposition_Tutorial.pdf)。更深入理解的额外资源包括 CS168 的第[7](https://web.stanford.edu/class/cs168/l/l7.pdf)、[8](http://theory.stanford.edu/~tim/s15/l/l8.pdf)和[9](https://web.stanford.edu/class/cs168/l/l9.pdf)讲，这些讲座对这些算法进行了高层次的讲解。在实际实现中，建议使用 Python 包（如 numpy、scipy 或 sklearn）中预编程的函数。虽然对大型语料库应用完整 SVD 可能消耗大量内存，但存在诸如 Truncated SVD 之类的可扩展技术，可以高效地提取前 $k$ 个向量分量。

### 绘制共现词嵌入

在这里，我们将使用大型电影评论数据集（Large Movie Review Dataset）。这是一个用于二分类情感分类的数据集，包含的数据量远超以前的基准数据集。我们提供了 25,000 条高度倾向的电影评论用于训练，以及 25,000 条用于测试。此外还有未标记的数据可供使用。下面我们提供了一个 `read_corpus` 函数，用于从数据集中提取电影评论文本。该函数还会在每个文档的开头和结尾添加 `<START>` 和 `<END>` 标记，并将单词转换为小写。你**不需要**执行任何其他形式的预处理。

In [ ]:
def read_corpus():
    """ Read files from the Large Movie Review Dataset.
        Params:
            category (string): category name
        Return:
            list of lists, with words from each of the processed files
    """
    files = imdb_dataset["train"]["text"][:NUM_SAMPLES]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]


我们来看看这些文档是什么样的……

In [ ]:
imdb_corpus = read_corpus()
pprint.pprint(imdb_corpus[:3], compact=True, width=100)
print("corpus size: ", len(imdb_corpus[0]))

### 问题 1.1：实现 `distinct_words` [代码]（2 分）

编写一个方法来计算语料库中出现的不同单词（词类型）。

你可以使用 `for` 循环来处理输入的 `corpus`（一个字符串列表的列表），但可以尝试使用 Python 列表推导式（通常更快）。特别是，[这个](https://coderwall.com/p/rcmaea/flatten-a-list-of-lists-in-one-line-in-python)技巧可能对展平列表的列表很有用。如果你不太熟悉 Python 列表推导式，这里提供了[更多信息](https://python-3-patterns-idioms-test.readthedocs.io/en/latest/Comprehensions.html)。

返回的 `corpus_words` 应该是有序的。你可以使用 Python 的 `sorted` 函数来实现。

你可能会发现使用 [Python 集合（set）](https://www.w3schools.com/python/python_sets.asp) 来去除重复单词很有用。

In [ ]:
def distinct_words(corpus):
    """ Determine a list of distinct words for the corpus.
        Params:
            corpus (list of list of strings): corpus of documents
        Return:
            corpus_words (list of strings): sorted list of distinct words across the corpus
            n_corpus_words (integer): number of distinct words across the corpus
    """
    corpus_words = []
    n_corpus_words = -1
    
    # ------------------
    # Write your implementation here.
    
    
    # ------------------

    return corpus_words, n_corpus_words

In [ ]:
# ---------------------
# Run this sanity check
# Note that this not an exhaustive check for correctness.
# ---------------------

# Define toy corpus
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
test_corpus_words, num_corpus_words = distinct_words(test_corpus)

# Correct answers
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
ans_num_corpus_words = len(ans_test_corpus_words)

# Test correct number of words
assert(num_corpus_words == ans_num_corpus_words), "Incorrect number of distinct words. Correct: {}. Yours: {}".format(ans_num_corpus_words, num_corpus_words)

# Test correct words
assert (test_corpus_words == ans_test_corpus_words), "Incorrect corpus_words.\nCorrect: {}\nYours:   {}".format(str(ans_test_corpus_words), str(test_corpus_words))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.2：实现 `compute_co_occurrence_matrix` [代码]（3 分）

编写一个方法，为特定的窗口大小 $n$（默认为 4）构建共现矩阵，考虑中心词前后各 $n$ 个词。在这里，我们开始使用 `numpy (np)` 来表示向量、矩阵和张量。如果你不熟悉 NumPy，cs231n [Python NumPy 教程](http://cs231n.github.io/python-numpy-tutorial/)的后半部分提供了一个 NumPy 教程。

In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """ Compute co-occurrence matrix for the given corpus and window_size (default of 4).
    
        Note: Each word in a document should be at the center of a window. Words near edges will have a smaller
              number of co-occurring words.
              
              For example, if we take the document "<START> All that glitters is not gold <END>" with window size of 4,
              "All" will co-occur with "<START>", "that", "glitters", "is", and "not".
    
        Params:
            corpus (list of list of strings): corpus of documents
            window_size (int): size of context window
        Return:
            M (a symmetric numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)): 
                Co-occurence matrix of word counts. 
                The ordering of the words in the rows/columns should be the same as the ordering of the words given by the distinct_words function.
            word2ind (dict): dictionary that maps word to index (i.e. row/column number) for matrix M.
    """
    words, n_words = distinct_words(corpus)
    M = None
    word2ind = {}
    
    # ------------------
    # Write your implementation here.
    
    
    # ------------------

    return M, word2ind

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# ---------------------

# Define toy corpus and get student's co-occurrence matrix
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)

# Correct M and word2ind
M_test_ans = np.array( 
    [[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,],
     [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,],
     [1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,],
     [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.,],
     [0., 0., 1., 0., 1., 1., 0., 0., 0., 1.,],
     [1., 0., 0., 1., 1., 0., 0., 0., 1., 0.,]]
)
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
word2ind_ans = dict(zip(ans_test_corpus_words, range(len(ans_test_corpus_words))))

# Test correct word2ind
assert (word2ind_ans == word2ind_test), "Your word2ind is incorrect:\nCorrect: {}\nYours: {}".format(word2ind_ans, word2ind_test)

# Test correct M shape
assert (M_test.shape == M_test_ans.shape), "M matrix has incorrect shape.\nCorrect: {}\nYours: {}".format(M_test.shape, M_test_ans.shape)

# Test correct M values
for w1 in word2ind_ans.keys():
    idx1 = word2ind_ans[w1]
    for w2 in word2ind_ans.keys():
        idx2 = word2ind_ans[w2]
        student = M_test[idx1, idx2]
        correct = M_test_ans[idx1, idx2]
        if student != correct:
            print("Correct M:")
            print(M_test_ans)
            print("Your M: ")
            print(M_test)
            raise AssertionError("Incorrect count at index ({}, {})=({}, {}) in matrix M. Yours has {} but should have {}.".format(idx1, idx2, w1, w2, student, correct))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.3：实现 `reduce_to_k_dim` [代码]（1 分）

构建一个方法，在矩阵上执行降维以产生 k 维嵌入。使用 SVD 取前 k 个分量，生成一个新的 k 维嵌入矩阵。

**注意：** numpy、scipy 和 scikit-learn (`sklearn`) 都提供了 SVD 的*某些*实现，但只有 scipy 和 sklearn 提供了 Truncated SVD 的实现，而且只有 sklearn 提供了计算大规模 Truncated SVD 的高效随机算法。因此，请使用 [sklearn.decomposition.TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)。

In [ ]:
def reduce_to_k_dim(M, k=2):
    """ Reduce a co-occurence count matrix of dimensionality (num_corpus_words, num_corpus_words)
        to a matrix of dimensionality (num_corpus_words, k) using the following SVD function from Scikit-Learn:
            - http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
    
        Params:
            M (numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)): co-occurence matrix of word counts
            k (int): embedding size of each word after dimension reduction
        Return:
            M_reduced (numpy matrix of shape (number of corpus words, k)): matrix of k-dimensioal word embeddings.
                    In terms of the SVD from math class, this actually returns U * S
    """    
    n_iters = 10    # Use this parameter in your call to `TruncatedSVD`
    M_reduced = None
    print("Running Truncated SVD over %i words..." % (M.shape[0]))
    
    # ------------------
    # Write your implementation here.
    
    
    # ------------------

    print("Done.")
    return M_reduced

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness 
# In fact we only check that your M_reduced has the right dimensions.
# ---------------------

# Define toy corpus and run student code
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)
M_test_reduced = reduce_to_k_dim(M_test, k=2)

# Test proper dimensions
assert (M_test_reduced.shape[0] == 10), "M_reduced has {} rows; should have {}".format(M_test_reduced.shape[0], 10)
assert (M_test_reduced.shape[1] == 2), "M_reduced has {} columns; should have {}".format(M_test_reduced.shape[1], 2)

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### 问题 1.4：实现 `plot_embeddings` [代码]（1 分）

在这里，你将编写一个函数，在二维空间中绘制一组二维向量。对于图形，我们将使用 Matplotlib (`plt`)。

对于这个例子，你可以参考[这段代码](http://web.archive.org/web/20190924160434/https://www.pythonmembers.club/2018/05/08/matplotlib-scatter-plot-annotate-set-text-at-label-each-point/)。以后，一个好的绘图方法是查看 [Matplotlib 画廊](https://matplotlib.org/gallery/index.html)，找到与你想要的图形类似的图，然后调整他们提供的代码。

In [ ]:
def plot_embeddings(M_reduced, word2ind, words):
    """ Plot in a scatterplot the embeddings of the words specified in the list "words".
        NOTE: do not plot all the words listed in M_reduced / word2ind.
        Include a label next to each point.
        
        Params:
            M_reduced (numpy matrix of shape (number of unique words in the corpus , 2)): matrix of 2-dimensioal word embeddings
            word2ind (dict): dictionary that maps word to indices for matrix M
            words (list of strings): words whose embeddings we want to visualize
    """

    # ------------------
    # Write your implementation here.
    
    
    # ------------------

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# The plot produced should look like the included file question_1.4_test.png 
# ---------------------

print ("-" * 80)
print ("Outputted Plot:")

M_reduced_plot_test = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1], [0, 0]])
word2ind_plot_test = {'test1': 0, 'test2': 1, 'test3': 2, 'test4': 3, 'test5': 4}
words = ['test1', 'test2', 'test3', 'test4', 'test5']
plot_embeddings(M_reduced_plot_test, word2ind_plot_test, words)

print ("-" * 80)

### 问题 1.5：共现图分析 [书面]（3 分）

现在我们将把你编写的所有部分整合在一起！我们将在大型电影评论语料库上计算固定窗口大小为 4（默认窗口大小）的共现矩阵。然后使用 TruncatedSVD 计算每个单词的二维嵌入。TruncatedSVD 返回 U\*S，所以我们需要对返回的向量进行归一化，使所有向量出现在单位圆附近（因此接近程度体现为方向上的接近）。**注意**：下面执行归一化的那行代码使用了 NumPy 的*广播*概念。如果你不了解广播，请查看
[数组计算：Jake VanderPlas 的广播讲解](https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html)。

运行下面的单元格来生成图形。运行可能需要几分钟。

In [ ]:
# -----------------------------
# Run This Cell to Produce Your Plot
# ------------------------------
imdb_corpus = read_corpus()
M_co_occurrence, word2ind_co_occurrence = compute_co_occurrence_matrix(imdb_corpus)
M_reduced_co_occurrence = reduce_to_k_dim(M_co_occurrence, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced_co_occurrence, axis=1)
M_normalized = M_reduced_co_occurrence / M_lengths[:, np.newaxis] # broadcasting

words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_normalized, word2ind_co_occurrence, words)

**验证你的图形是否与作业压缩包中的 "question_1.5.png" 匹配。如果不匹配，请使用 "question_1.5.png" 中的图形来回答接下来的两个问题。**

a. 在二维嵌入空间中找出至少两组聚集在一起的词。对你观察到的每个聚类给出解释。

#### <font color="red">在此处写下你的答案。</font>


b. 有哪些你认为本应聚集在一起但实际上并没有的？请描述至少两个例子。

#### <font color="red">在此处写下你的答案。</font>

## 第 2 部分：基于预测的词向量（15 分）

正如课堂上所讨论的，最近基于预测的词向量（如 word2vec 和 GloVe）表现出了更好的性能（GloVe 也利用了计数的优势）。在这里，我们将探索 GloVe 生成的嵌入。请重新查阅课堂笔记和讲义幻灯片，以了解有关 word2vec 和 GloVe 算法的更多细节。如果你喜欢挑战，可以尝试阅读 [GloVe 的原始论文](https://nlp.stanford.edu/pubs/glove.pdf)。

然后运行以下单元格将 GloVe 向量加载到内存中。**注意**：如果这是你第一次运行这些单元格，即下载嵌入模型，将需要几分钟时间。如果你之前运行过这些单元格，再次运行将直接加载模型而无需重新下载，大约需要 1 到 2 分钟。

In [ ]:
def load_embedding_model():
    """ Load GloVe Vectors
        Return:
            wv_from_bin: All 400000 embeddings, each length 200
    """
    import gensim.downloader as api
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print("Loaded vocab size %i" % len(list(wv_from_bin.index_to_key)))
    return wv_from_bin
wv_from_bin = load_embedding_model()

#### 注意：如果你收到 "reset by peer" 错误，请重新运行单元格以重启下载。

### 降低词嵌入的维度
让我们直接比较 GloVe 嵌入与共现矩阵嵌入。为了避免内存不足，我们将使用 40000 个 GloVe 向量的样本。
运行以下单元格以：

1. 将 40000 个 GloVe 向量放入矩阵 M 中
2. 运行 `reduce_to_k_dim`（你的 Truncated SVD 函数）将向量从 200 维降到 2 维。

In [ ]:
def get_matrix_of_vectors(wv_from_bin, required_words):
    """ Put the GloVe vectors into a matrix M.
        Param:
            wv_from_bin: KeyedVectors object; the 400000 GloVe vectors loaded from file
        Return:
            M: numpy matrix shape (num words, 200) containing the vectors
            word2ind: dictionary mapping each word to its row number in M
    """
    import random
    words = list(wv_from_bin.index_to_key)
    print("Shuffling words ...")
    random.seed(225)
    random.shuffle(words)
    print("Putting %i words into word2ind and matrix M..." % len(words))
    word2ind = {}
    M = []
    curInd = 0
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    for w in required_words:
        if w in words:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    M = np.stack(M)
    print("Done.")
    return M, word2ind

In [ ]:
# -----------------------------------------------------------------
# Run Cell to Reduce 200-Dimensional Word Embeddings to k Dimensions
# Note: This should be quick to run
# -----------------------------------------------------------------
M, word2ind = get_matrix_of_vectors(wv_from_bin, words)
M_reduced = reduce_to_k_dim(M, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced, axis=1)
M_reduced_normalized = M_reduced / M_lengths[:, np.newaxis] # broadcasting

**注意：如果你在本地机器上遇到内存不足的问题，请尝试关闭其他应用程序以释放更多内存。你可能需要重新启动机器以释放额外内存，然后立即运行 jupyter notebook，看看是否能够正确加载词向量。如果在这些尝试之后仍然无法在本地机器上加载嵌入，请前往答疑时间或联系课程工作人员。**

### 问题 2.1：GloVe 图分析 [书面]（3 分）

运行以下单元格以绘制 `['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']` 的二维 GloVe 嵌入。

In [ ]:
words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_reduced_normalized, word2ind, words)

**验证你的图形是否与作业压缩包中的 "question_2.1.png" 匹配。如果不匹配，请使用 "question_2.1.png"（如果适用，也使用 "question_1.5.png"）来回答接下来的两个问题。**

a. 这个图与之前从共现矩阵生成的图有什么不同之处？有什么相同之处？

#### <font color="red">在此处写下你的答案。</font>

b. 为什么 GloVe 图（question_2.1.png）可能与之前从共现矩阵生成的图（question_1.5.png）有所不同？

#### <font color="red">在此处写下你的答案。</font>

### 余弦相似度
现在我们有了词向量，我们需要一种方法来量化单个词之间的相似度。一种这样的度量是余弦相似度。我们将使用它来找出彼此"接近"和"远离"的词。

我们可以将 n 维向量视为 n 维空间中的点。从这个角度看，[L1](http://mathworld.wolfram.com/L1-Norm.html) 和 [L2](http://mathworld.wolfram.com/L2-Norm.html) 距离有助于量化"我们需要行进"的空间量以到达这两个点之间。另一种方法是考察两个向量之间的角度。从三角学中我们知道：

<img src="./imgs/inner_product.png" width=20% style="float: center;"></img>

与其计算实际角度，我们可以将相似度表示为 $similarity = cos(\Theta)$。形式上，两个向量 $p$ 和 $q$ 之间的[余弦相似度](https://en.wikipedia.org/wiki/Cosine_similarity) $s$ 定义为：

$$s = \frac{p \cdot q}{||p|| ||q||}, \textrm{ 其中 } s \in [-1, 1] $$ 

### 问题 2.2：具有多重含义的词（1.5 分）[代码 + 书面] 
多义词和同形异义词是指具有不止一种含义的词语（请查看此[维基页面](https://en.wikipedia.org/wiki/Polysemy)以了解多义词和同形异义词之间的区别）。找到一个具有*至少两种不同含义*的词，使得前 10 个最相似的词（根据余弦相似度）包含来自*两种*含义的相关词。例如，"leaves" 既有"离开"的含义，也有"植物的叶子"的含义出现在前 10 名中；"scoop" 既有"蛋筒"的含义，也有"内幕消息"的含义。你可能需要尝试多个多义或同形异义词才能找到一个。

请说明你发现的词以及出现在前 10 名中的多种含义。你认为为什么许多你尝试过的多义词或同形异义词不起作用（即前 10 个最相似的词只包含该词的**一种**含义）？

**注意**：你应该使用 `wv_from_bin.most_similar(word)` 函数来获取前 10 个最相似的词。该函数根据与给定词的余弦相似度对词汇表中的所有其他词进行排序。如需进一步帮助，请查看 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.most_similar)__。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.3：同义词和反义词（2 分）[代码 + 书面] 

在考虑余弦相似度时，通常更方便考虑余弦距离，它只是 1 - 余弦相似度。

找到三个词 $(w_1,w_2,w_3)$，其中 $w_1$ 和 $w_2$ 是同义词，$w_1$ 和 $w_3$ 是反义词，但余弦距离 $(w_1,w_3) <$ 余弦距离 $(w_1,w_2)$。

例如，$w_1$="happy" 与 $w_3$="sad" 的距离比与 $w_2$="cheerful" 更近。请找出一个满足上述条件的不同示例。一旦找到你的示例，请给出一个可能的解释，说明为什么会出现这种反直觉的结果。

你应该在这里使用 `wv_from_bin.distance(w1, w2)` 函数来计算两个词之间的余弦距离。请查看 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.distance)__ 以获取进一步帮助。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.4：词向量类比 [书面]（1.5 分）
词向量已经被证明*有时*能够解决类比问题。

例如，对于类比 "man : grandfather :: woman : x"（读作：man 对于 grandfather 正如 woman 对于 x），x 是什么？

在下面的单元格中，我们演示如何使用 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar)__ 中的 `most_similar` 函数通过词向量来找到 x。该函数找到与 `positive` 列表中的词最相似、与 `negative` 列表中的词最不相似的词（同时省略输入词，因为它们通常是最相似的；参见[这篇论文](https://www.aclweb.org/anthology/N18-2039.pdf)）。类比的答案将具有最高的余弦相似度（最大的返回数值）。

In [ ]:
# Run this cell to answer the analogy -- man : grandfather :: woman : x
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'grandfather'], negative=['man']))

设 $m$、$g$、$w$ 和 $x$ 分别表示 `man`、`grandfather`、`woman` 和答案的词向量。在回答中仅使用向量 $m$、$g$、$w$ 以及向量算术运算符 $+$ 和 $-$，我们是在最大化与 $x$ 的余弦相似度的表达式是什么？

提示：回想一下，词向量只是表示一个词的多维向量。尝试绘制一个二维示例，使用每个向量的任意位置可能会有所帮助。`man` 和 `woman` 相对于 `grandfather` 和答案在坐标平面中的位置如何？

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.5：寻找类比 [代码 + 书面]（1.5 分）
a. 对于前面的例子，很明显 "grandmother" 完成了这个类比。但请给出一个直观的解释，说明为什么 `most_similar` 函数会给我们类似于 "granddaughter"、"daughter" 或 "mother" 这样的词？

#### <font color="red">在此处写下你的答案。</font>

b. 找到一个根据这些向量成立的类比示例（即目标词排名第一）。在你的解答中，请以 x:y :: a:b 的形式给出完整的类比。如果你认为该类比比较复杂，请用一两句话解释为什么该类比成立。

**注意**：你可能需要尝试多个类比才能找到一个有效的！

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------

# Test the solution
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] == b

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.6：错误的类比 [代码 + 书面]（1.5 分）
a. 下面，我们期望看到类比 "hand : glove :: foot : **sock**"，但得到了一个意外的结果。请给出一个潜在的原因，说明为什么这个特定的类比会得出这样的结果？

In [ ]:
pprint.pprint(wv_from_bin.most_similar(positive=['foot', 'glove'], negative=['hand']))

#### <font color="red">在此处写下你的答案。</font>

b. 找到另一个根据这些向量*不*成立的类比示例。在你的解答中，以 x:y :: a:b 的形式说明预期的类比，并根据词向量指出 b 的**错误**值（在上一个例子中，这将是 **'45,000-square'**）。

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] != b

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.7：词向量中的偏见引导分析 [书面]（1 分）

意识到我们的词嵌入中隐含的偏见（性别、种族、性取向等）非常重要。偏见是危险的，因为它可以通过使用这些模型的应用程序强化刻板印象。

运行下面的单元格，检查 (a) 哪些词与 "man" 和 "profession" 最相似、与 "woman" 最不相似，以及 (b) 哪些词与 "woman" 和 "profession" 最相似、与 "man" 最不相似。指出与女性相关的词列表和与男性相关的词列表之间的差异，并解释这如何反映了性别偏见。

In [ ]:
# Run this cell
# Here `positive` indicates the list of words to be similar to and `negative` indicates the list of words to be
# most dissimilar from.

pprint.pprint(wv_from_bin.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'profession'], negative=['man']))

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.8：词向量中的偏见独立分析 [代码 + 书面]（1 分）

使用 `most_similar` 函数找到另一对比类，展示向量表现出的某种偏见。请简要解释你发现的偏见示例。

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">在此处写下你的答案。</font>

### 问题 2.9：思考偏见 [书面]（2 分）

a. 给出一个偏见如何进入词向量的解释。简要描述一个展示这种偏见来源的真实世界示例。你的真实世界示例应该专注于词向量，而不是其他 AI 系统中的偏见（例如，ChatGPT）。

#### <font color="red">在此处写下你的答案。</font>

b. 你可以使用什么方法来减轻词向量表现出的偏见？简要描述一个展示该方法的真实世界示例。


#### <font color="red">在此处写下你的答案。</font>

# <font color="blue"> 提交说明</font>

1. 点击 Jupyter Notebook 顶部的 Save 按钮。
2. 选择 Cell -> All Output -> Clear。这将清除所有单元格的输出（但会保留所有单元格的内容）。
3. 选择 Cell -> Run All。这将按顺序运行所有单元格，需要几分钟时间。
4. 重新运行所有内容后，选择 File -> Download as -> PDF via LaTeX（如果使用 "PDF via LaTex" 遇到问题，你也可以将网页保存为 PDF。<font color='blue'> 确保你的所有解答（特别是代码部分）都显示在 PDF 中</font>，如果提供的代码由于代码单元格中的换行问题而被截断，这是可以接受的）。
5. 查看 PDF 文件，确保你的所有解答都正确显示。评分人员只会看到 PDF！
6. 在 Gradescope 上提交你的 PDF。